<a href="https://colab.research.google.com/github/christinengalle19-collab/Assignement2New/blob/main/Assignment_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##For this assignment, I selected a text dataset from Project Gutenberg. It provides public domain books that are legally accessible and easy to preprocess. The dataset is lightweight, suitable for training a simple generative model, and allows clear evaluation of text coherence, vocabulary patterns, and stylistic reproduction.

##Description of GPTs architecture and its functionality.
##GPT (Generative Pre trained Transformer) is a deep learning architecture based on the Transformer model, which uses layers of self attention and feed forward neural networks to process and generate text. It is trained on vast amounts of text data to learn patterns of language, grammar, and context. Functionally, GPT predicts the next word in a sequence by analyzing the relationships between all previous words, enabling it to produce coherent, context aware responses. Its architecture allows it to handle long range dependencies in text, making it powerful for tasks like translation, summarization, and conversational AI.

In [36]:
!pip install transformers torch
!pip install requests
!pip install torch
!pip install transformers


In [37]:
import numpy as np

import requests
import re
from transformers import pipeline, set_seed
import torch
import tensorflow as tf
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Embedding

In [38]:
#load the text from gutenberg project
url = "https://www.gutenberg.org/files/11/11-0.txt"
response = requests.get(url)
text = response.text
print(text[:500])

*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The


In [39]:
#clean the text byremoving header and footer
text = re.sub(r'\[.*?\]', '', text)
text = text.strip()
print(text[:500])


*** START OF THE PROJECT GUTENBERG EBOOK 11 ***






Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s


In [40]:
#save the text in a file
with open("shakespeare.txt", "w", encoding="utf-8") as file:
    file.write(text)
    print("Text saved to shakespeare.txt")

Text saved to shakespeare.txt


In [41]:
## Load the pre-trained model and tokenizer
#Next, we'll load a pre-trained GPT-2 model and its associated tokenizer. The tokenizer is responsible for converting text into tokens that the model can understand.

model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [42]:
#tokenizing the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_index = len(tokenizer.word_index) + 1
print(total_index)

3066


In [43]:
# Analyze the vocabulary
#top 20 frequent words
sorted_words = sorted(tokenizer.word_counts.items(), key=lambda x: x[1], reverse=True)
top_words = sorted_words[:20]
print("Top 20 frequent words:")
for word, count in top_words:
    print(f"{word}: {count}")
    print("\n")


Top 20 frequent words:
the: 1623


”: 1040


and: 795


to: 719


a: 621


she: 535


of: 502


it: 494


said: 459


alice: 385


in: 361


was: 356


you: 319


i: 273


that: 264


as: 254


her: 248


at: 206


on: 192


with: 179




In [44]:
#split for training
sequences = []
for line in text.split('\n'):
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        n_gram_sequence = tokens[:i+1]
sequences = tokenizer.texts_to_sequences([text])
sequences = np.array(sequences)
max_length = max([len(seq) for seq in sequences])
sequences = pad_sequences(sequences, maxlen=max_length, padding='pre')
x = sequences[:, :-1]
y = sequences[:, -1]
y = to_categorical(y, num_classes=total_index)
print(x.shape)
print(y.shape)


(1, 27763)
(1, 3066)


In [45]:
#Modele LSTM
model = Sequential()
model.add(Embedding(total_index, 10, input_length=max_length-1))
model.add(LSTM(50))
model.add(Dense(total_index, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x, y, epochs=100, verbose=1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.0000e+00 - loss: 8.0288
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 8.0253
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 1.0000 - loss: 8.0217
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 1.0000 - loss: 8.0179
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 1.0000 - loss: 8.0138
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 8.0094
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 1.0000 - loss: 8.0044
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 1.0000 - loss: 7.9987
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 1.0000 - loss: 7.9921
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 7.9843
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 7.9749
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 7.9633
Epoch 1

In [46]:
## Define a function for text generation
#This function will be our main tool for generating text. It takes a prompt and several parameters that control the generation process.


def generate_text(prompt, max_length=100, temperature=0.7, num_return_sequences=1, top_k=50, top_p=0.95):
    # Encode the prompt and get attention mask
    inputs = tokenizer(prompt, return_tensors='pt', padding=True)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True, # Enable sampling for temperature to take effect
        top_k=top_k, # Pass top_k parameter
        top_p=top_p # Pass top_p parameter
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [47]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Save the current Keras tokenizer and model instances
original_keras_tokenizer = tokenizer
original_keras_model = model

# Re-initialize the GPT2Tokenizer and GPT2LMHeadModel for the generate_text function
# This ensures the correct tokenizer and model are used for the generation process.
model_name = "gpt2"
openai_tokenizer = GPT2Tokenizer.from_pretrained(model_name)
openai_model = GPT2LMHeadModel.from_pretrained(model_name)

# Set the pad token for the GPT2Tokenizer
# This is necessary because the tokenizer call with `padding=True` requires a pad token.
openai_tokenizer.pad_token = openai_tokenizer.eos_token

# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel to the global variables
tokenizer = openai_tokenizer
model = openai_model

# Call the generate function
#prompt 1
generated_text_output = generate_text('Hello Word')
print(generated_text_output)

# Restore the Keras tokenizer and model to the global variables
# to avoid breaking other parts of the notebook that might rely on them.
tokenizer = original_keras_tokenizer
model = original_keras_model

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Hello Word is that there are many people who use this tool on their own and that this tool is not meant to be used for anyone else. Please read the disclaimer before using it.

The following is a list of links that you can use to help with the installation process.

Download and install the following binaries, as well as their dependencies, in your Windows system:

Windows 7 64 bit

Windows 8 64 bit

Windows 10 64 bit

Windows Vista 64


In [48]:
# Save the current global Keras tokenizer and model instances
# These variables ('original_keras_tokenizer', 'original_keras_model') should exist from cell 'aAA4J2n8BS4f'.
# No need to re-save them as they represent the original Keras objects.

# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel (loaded in aAA4J2n8BS4f)
# to the global 'tokenizer' and 'model' variables for this function call.
# We assume 'openai_tokenizer' and 'openai_model' were correctly defined and 'openai_tokenizer.pad_token' was set in 'aAA4J2n8BS4f'.
temp_saved_tokenizer = tokenizer
temp_saved_model = model

tokenizer = openai_tokenizer
model = openai_model

# Call the generate function
#prompt2
generated_text_output = generate_text('I really like ai')
print(generated_text_output)

# Restore the original global tokenizer and model (Keras instances)
tokenizer = temp_saved_tokenizer
model = temp_saved_model

I really like aiun aiun, but I just don't like it when people like me say it's like 'oh my god it's like I can't believe it'.


It's kind of like a 'I'm sorry, I didn't mean it' kind of thing, like 'yeah, I can't believe you were so mean to me. I don't really know what to say anymore'.


I have a lot of respect for you, but I also


In [49]:
# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel (loaded in aAA4J2n8BS4f)
# to the global 'tokenizer' and 'model' variables for this function call.
# We assume 'openai_tokenizer' and 'openai_model' were correctly defined and 'openai_tokenizer.pad_token' was set in 'aAA4J2n8BS4f'.
temp_saved_tokenizer = tokenizer
temp_saved_model = model

tokenizer = openai_tokenizer
model = openai_model

# Call the generate function
prompt3= generate_text('willis college is the place to do BIA program')
print("Exercise 3: Adjusting temperature")
print("Low temperature (0.2):")
print(generate_text(prompt3, temperature=0.2, max_length=200)) # Increased max_length
print("\nHigh temperature (1.5):")
print(generate_text(prompt3, temperature=1.5, max_length=200)) # Increased max_length
print("\n" + "="*50 + "\n")

# Restore the original global tokenizer and model (Keras instances)
tokenizer = temp_saved_tokenizer
model = temp_saved_model

Exercise 3: Adjusting temperature
Low temperature (0.2):
willis college is the place to do BIA program research. Here are some links:

What is BIA?

BIA is a national program that provides a high-level level of academic research, including clinical studies, to the general public and to all students enrolled in college and university programs.

BIA is a national program that provides a high-level level of academic research, including clinical studies, to the general public and to all students enrolled in college and university programs. BIA is a national program that provides a high-level of academic research, including clinical studies, to the general public and to all students enrolled in college and university programs. BIA is a national program that provides a high-level of academic research, including clinical studies, to the general public and to all students enrolled in college and university programs.

BIA is a national program that provides a high-level of academic research, in

In [50]:
# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel (loaded in aAA4J2n8BS4f)
# to the global 'tokenizer' and 'model' variables for this function call.
# We assume 'openai_tokenizer' and 'openai_model' were correctly defined and 'openai_tokenizer.pad_token' was set in 'aAA4J2n8BS4f'.
temp_saved_tokenizer = tokenizer
temp_saved_model = model

tokenizer = openai_tokenizer
model = openai_model
#call the generation functiom
prompt4 = generate_text("In the year 2050, transportation will")
print("Exercise 4: Using top_k and top_p")
print("Default settings:")
print(generate_text(prompt4, max_length=200)) # Increased max_length
print("\nLow top_k and top_p:")
print(generate_text(prompt4, top_k=10, top_p=0.5, max_length=200)) # Increased max_length
print("\n" + "="*50 + "\n")


Exercise 4: Using top_k and top_p
Default settings:
In the year 2050, transportation will be the most important economic activity of the 21st century. This is because of the huge benefits it will bring to the people of the world. With transportation, we can be sure that the world will be better off.

This is the real challenge we face. The problems of the 21st century will be solved by the use of the road. To achieve this, we need to build a national transportation system that is sustainable and sustainable in every possible way. We can do this by improving the roads, but we need to do it in a way that protects the people, as opposed to the traffic. We need to ensure that the roads are not built with a high risk of accidents, as they do in a modern world, where a high cost of living means that people have to make decisions based on where they live.

The biggest challenge facing the 21st century is our inability to bring people together. In the 21st century, the roads are

Low top_k and

In [ ]:
# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel (loaded in aAA4J2n8BS4f)
# to the global 'tokenizer' and 'model' variables for this function call.
# We assume 'openai_tokenizer' and 'openai_model' were correctly defined and 'openai_tokenizer.pad_token' was set in 'aAA4J2n8BS4f'.
temp_saved_tokenizer = tokenizer
temp_saved_model = model

tokenizer = openai_tokenizer
model = openai_model
#call the generation functiom

#Exercise 5
prompt5= generate_text('AI system will always needs human intervention')
print("Exercise 5: Prompt engineering")
base_prompt = "Write a short story about"
topics = ["a robot learning to paint", "a time traveler's first day in the past", "a sentient AI's ethical dilemma"]

for topic in topics:
    full_prompt = f"{base_prompt} {topic}. The story should be creative and engaging:"
    print(f"Topic: {topic}")
    print(generate_text(full_prompt, max_length=200))
    print("\n" + "-"*150 + "\n")

Exercise 5: Prompt engineering
Topic: a robot learning to paint
Write a short story about a robot learning to paint. The story should be creative and engaging: the writer will create an interactive story around the process, but the story will also be written by the artist and will be unique to each story. The writer should also include a video or audio commentary to explain the story. The story should not be an in-joke or a joke, but rather a great story that shows how a robot can learn to do some of the things a human can't.

When designing a story, your goal is to present the reader with the best possible experience possible. So you should always have a clear goal in mind:

What are the most important things to know about a robot?

What are the major challenges of a robot's life and how can they be overcome?

Do you think your story should be about a robot learning to paint, or about how a human can learn to paint?

How can you make a story that

-------------------------------------